In [2]:
import xarray as xr
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
import numpy as np
import matplotlib.pyplot as plt


In [2]:
msg_file = '/mnt/data8tb/fire_detection/msg/L1b/MSG3-SEVI-MSG15-0100-NA-20230930171241.955000000Z-NA.nat'
msg_cloud = '/mnt/data8tb/fire_detection/msg/L1b/MSG3-SEVI-MSGCLMK-0100-0100-20230930171500.000000000Z-NA.grb'
modis_file = '/mnt/data8tb/fire_detection/modis/MOD/L1b/MOD021KM.A2024165.1110.061.2024165191849.hdf'
modis_cloud = '/mnt/data8tb/fire_detection/modis/MOD/CM/MOD35_L2.A2023273.1105.061.2023273191143.hdf'

In [27]:
modis_cloud = '/mnt/data8tb/fire_detection/modis/CM/MYD35_L2.A2020170.1340.061.2020171151702.hdf'

In [19]:
MODIS_WAVELENGTHS = {
    "1": 0.645, 
    "2": 0.8585,
    "3": 0.469,
    "4": 0.555,
    "5": 1.24,
    "6": 1.64,
    "7": 2.13,
    "8": 0.4125,
    "9": 0.443,
    "10": 0.488,
    "11": 0.531,
    "12": 0.551,
    "13lo": 0.667,
    "13hi": 0.667,
    "14lo": 0.678,
    "14hi": 0.678,
    "15": 0.748,
    "16": 0.8695,
    "17": 0.905,
    "18": 0.936,
    "19": 0.940,
    "20": 3.75,
    "21": 3.959,
    "22": 3.959,
    "23": 4.05,
    "24": 4.4655,
    "25": 4.5155,
    "26": 1.375,
    "27": 6.715,
    "28": 7.325,
    "29": 8.55,
    "30": 9.73,
    "31": 11.03,
    "32": 12.02,
    "33": 13.335,
    "34": 13.635,
    "35": 13.935,
    "36": 14.235,
}

In [4]:
# MODIS Radiances
file = [modis_file]
scn = Scene(
            reader="modis_l1b",
            filenames=file
        )

# Load radiance bands
# channels = get_modis_channel_numbers() # This assigns the wrong order of bands
channels = list(MODIS_WAVELENGTHS.keys())
scn.load(channels, generate=False, calibration='radiance')

In [33]:
# MODIS CM
file = [modis_cloud]
scn = Scene(
            reader="modis_l2",
            filenames=file
        )
# Load cloud mask data
datasets = scn.available_dataset_names()
# Needs to be loaded at 1000 m resolution for all channels to match
scn.load(['cloud_mask', 'latitude', 'longitude'])

latitudes_cm = scn['latitude']
longitudes_cm = scn['longitude']

latitudes_cm.values.max(), latitudes_cm.values.min(), longitudes_cm.values.max(), longitudes_cm.values.min()

(50.876537, 29.425581, 6.125246, -26.96419)

In [34]:
from pyresample import create_area_def

area_extent = [longitudes_cm.values.min(), latitudes_cm.values.min(), longitudes_cm.values.max(), latitudes_cm.values.max()]  # [min_lon, min_lat, max_lon, max_lat]

area_def = create_area_def('my_area',
                           {'proj': 'longlat', 'datum': 'WGS84'},
                           area_extent=area_extent,
                           resolution=0.027,
                           units='degrees',
                           description='Gloxx bal degree lat-lon grid')

Rounding shape to (795, 1226) and resolution from (0.026999999999999247, 0.026999999999993918) meters to (0.026989751694447454, 0.02698233502465974) meters


In [35]:
new_scn = scn.resample(area_def)
ds_resampled = new_scn.to_xarray_dataset()

In [36]:
ds_resampled

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*
Attributes:
    start_time:           2020-06-18 13:40:00
    reader:               modis_l2
    ancillary_variables:  []
    end_time:             2020-06-18 13:45:00
    modifiers:            ()
    sensor:               modis
    platform_name:        EOS-Aqua

In [5]:
from pyresample import create_area_def

area_def = create_area_def('my_area',
                           {'proj': 'longlat', 'datum': 'WGS84'},
                           area_extent=[-10.019531, 30.22889, 46.617188, 49.012224],
                           resolution=0.027,
                           units='degrees',
                           description='Gloxx bal degree lat-lon grid')

Rounding shape to (696, 2098) and resolution from (0.027000000000001023, 0.027000000000001023) meters to (0.02699557626310772, 0.02698754885057472) meters


In [6]:
new_scn = scn.resample(area_def)

In [ ]:
ds_resampled = new_scn.to_xarray_dataset()

In [8]:
ds_resampled

<xarray.Dataset> Size: 222MB
Dimensions:  (y: 696, x: 2098)
Coordinates:
    crs      object 8B GEOGCRS["unknown",DATUM["World Geodetic System 1984",E...
  * y        (y) float64 6kB 49.0 48.97 48.94 48.92 ... 30.32 30.3 30.27 30.24
  * x        (x) float64 17kB -10.01 -9.979 -9.952 -9.925 ... 46.55 46.58 46.6
Data variables: (12/38)
    2        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    6        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    5        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    20       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    14hi     (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    24       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    ...       ...
    16       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    23       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    31       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    12       (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    3        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
    1        (y, x) float32 6MB dask.array<chunksize=(696, 2098), meta=np.ndarray>
Attributes: (12/15)
    start_time:           2024-06-13 11:10:00
    reader:               modis_l1b
    ancillary_variables:  []
    calibration:          radiance
    modifiers:            ()
    end_time:             2024-06-13 11:15:00
    ...                   ...
    standard_name:        toa_outgoing_radiance_per_unit_wavelength
    file_type:            hdf_eos_data_1000m
    sensor:               modis
    units:                Watts/m^2/micrometer/steradian
    platform_name:        EOS-Terra
    rows_per_scan:        10

In [42]:
# MODIS cloud mask
scn = Scene(
            reader="modis_l2",
            filenames=[modis_cloud]
        )
# Load cloud mask data
datasets = scn.available_dataset_names()
# Needs to be loaded at 1000 m resolution for all channels to match
scn.load(datasets, generate=False, resolution=1000)

latitudes_cm = scn['latitude']
longitudes_cm = scn['longitude']

latitudes_cm.values.max(), latitudes_cm.values.min(), longitudes_cm.values.max(), longitudes_cm.values.min()

(50.876537, 29.425581, 6.125246, -26.96419)

In [ ]:
area_extent = [longitudes_cm.values.min(), latitudes_cm.values.min(), longitudes_cm.values.max(), latitudes_cm.values.max()]  # [min_lon, min_lat, max_lon, max_lat]

area_def = create_area_def('my_area',
                           {'proj': 'longlat', 'datum': 'WGS84'},
                           area_extent=area_extent,
                           resolution=0.027,
                           units='degrees',
                           description='Gloxx bal degree lat-lon grid')

In [38]:
new_scn_cloud = scn.resample(area_def)

In [39]:
ds_resampled_clouds = new_scn_cloud.to_xarray_dataset()

In [40]:
ds_resampled_clouds

<xarray.Dataset> Size: 2MB
Dimensions:            (y: 795, x: 1226)
Coordinates:
    crs                object 8B GEOGCRS["unknown",DATUM["World Geodetic Syst...
  * y                  (y) float64 6kB 50.86 50.84 50.81 ... 29.49 29.47 29.44
  * x                  (x) float64 10kB -26.95 -26.92 -26.9 ... 6.085 6.112
Data variables:
    cloud_mask         (y, x) uint8 975kB dask.array<chunksize=(795, 1226), meta=np.ndarray>
    quality_assurance  (y, x) uint8 975kB dask.array<chunksize=(795, 1226), meta=np.ndarray>
Attributes:
    start_time:           2020-06-18 13:40:00
    reader:               modis_l2
    ancillary_variables:  []
    end_time:             2020-06-18 13:45:00
    modifiers:            ()
    resolution:           1000
    platform_name:        EOS-Aqua
    rows_per_scan:        10
    sensor:               modis

In [41]:
# For a single variable (DataArray):
da = ds_resampled_clouds['cloud_mask']  # replace with the band/var you want
da.rio.to_raster('/home/sgirtsou/Projects/rs_tools/cm_satpy.tif')

In [43]:
msg_patch = '/mnt/nvme2tb/patched/patched/20210518134243_patch_311.tif'

In [47]:
msg = xr.open_dataset(msg_patch)

In [56]:
msg['band_data'][13].shape

(32, 32)

In [57]:
msg['band_data'][12].shape

(32, 32)

In [58]:
msg

<xarray.Dataset> Size: 115kB
Dimensions:      (band: 14, x: 32, y: 32)
Coordinates:
  * band         (band) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * x            (x) float64 256B -2.61e+05 -2.58e+05 ... -1.71e+05 -1.68e+05
  * y            (y) float64 256B 3.871e+06 3.874e+06 ... 3.961e+06 3.964e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (band, y, x) float64 115kB ...

In [59]:
import numpy as np
import xarray as xr
from pyresample import geometry, create_area_def
from pyresample.kd_tree import resample_nearest
from rasterio.transform import from_bounds
import rioxarray

In [60]:
band_data = msg['band_data']           # (bands, y, x)
lon2d = band_data[13].astype(np.float64)
lat2d = band_data[12].astype(np.float64)

In [63]:
bad = ~np.isfinite(lon2d) | ~np.isfinite(lat2d)


In [64]:
swath = geometry.SwathDefinition(lons=lon2d, lats=lat2d)

In [65]:
min_lon, max_lon = np.nanmin(lon2d), np.nanmax(lon2d)
min_lat, max_lat = np.nanmin(lat2d), np.nanmax(lat2d)
height, width = lat2d.shape 

In [68]:
target = create_area_def(
    'wgs84_patch',
    projection= {'proj': 'laea', 'lat_0': -90, 'lon_0': 0, 'a': 6371228.0, 'units': 'm'}, datum='WGS84', units='degrees',
    area_extent=[float(min_lon), float(min_lat), float(max_lon), float(max_lat)],
    shape=(height, width),
)

In [69]:
# radius_of_influence is in meters when using lat/lon; 30 km is fine for MSG patch.
roi_m = 30000
fill = np.nan

In [70]:
out = np.empty((band_data.shape[0], height, width), dtype=np.float32)
for b in range(band_data.shape[0]):
    band = band_data[b].astype(np.float32)
    # (optional) mask band where geolocation invalid
    band = np.where(bad, np.nan, band)
    out[b] = resample_nearest(
        source_geo_def=swath,
        target_geo_def=target,
        data=band,
        radius_of_influence=roi_m,
        epsilon=0.5,
        fill_value=fill,
    )

In [71]:
da = xr.DataArray(out, dims=('band', 'y', 'x'))

# Build transform from target extent
transform = from_bounds(min_lon, min_lat, max_lon, max_lat, width=width, height=height)

da = da.rio.write_crs("EPSG:4326", inplace=False)
da = da.rio.write_transform(transform, inplace=False)
da = da.rio.write_nodata(np.nan, encoded=True)

# Save multi-band GeoTIFF
da.rio.to_raster("/home/sgirtsou/Projects/rs_tools/msg_patch_wgs84.tif")
